# colony_grid

> Phase 18 (`specs/multiagent/phase18/`): the colony-rule cells and plans. `COLONY_ARMS` (plain, confirm, colony, colony with guards, the public track record), `colonyCell` (two holders per clue, a quarter of the agents faulty), Part 0's numpy grid, the pilot, Stage 1 and Stage 2 grids, call estimates, the pre-set cost rules, `goNoGo`, `COLONY_PLAN` and `COLONY2_PLAN`; `syntheticRows18` for tests.

In [ ]:
#| default_exp colony_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## Arms and cells

In [ ]:
#| export
import copy

import numpy as np

from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueCell
from explRL.colony_report import NAN, cellVals, pairs, signFlipExact
from explRL.fix_grid import fixEstimate
from explRL.llm_budget import PRICES
from explRL.llm_grid import HAIKU

WORLD18 = {**{k: CLUE_SETTINGS[k] for k in BOARD}, 'r': 2}
COLONY_ARMS = dict(plain={}, confirm=dict(confirm=True), colony=dict(colony=True),
                   colonyGuards=dict(colony=True, guards='quarter'), record=dict(record=True))
PART0_ARMS = {**{k: v for k, v in COLONY_ARMS.items() if k != 'record'},              # numpy: rule agents read no record
              **{f'colonyTheta{th}': dict(colony=True, theta=th) for th in (1, 2, 4)}}
COLONY_SEEDS, PILOT_SEED, PART0_SEEDS = list(range(1800, 1808)), 1890, list(range(3000, 3030))
STAGE1_N, STAGE2_N, PART0_N = [8, 16], [32, 64], [4, 8, 16, 32, 64, 128]
FAULTS18 = ('clean', 'independent', 'collude', 'error')


def colonyCell(study, arm, N, seed, fault='clean', runner='llm', q=2, arms=None):
    """One Phase 18 cell, level '<arm>|<fault>': WORLD18 plus the arm's options (guards 'quarter' -> N // 4); with a fault,
    the first N // 4 slots are faulty members of that kind; q != 2 (Part 0) is stored in the world and the level."""
    w = {**WORLD18, **{k: (N // 4 if v == 'quarter' else v) for k, v in (arms or {**COLONY_ARMS, **PART0_ARMS})[arm].items()}}
    if fault in ('collude', 'error'):
        w['fault'] = fault
    if q != 2:
        w.update(q=q, r=q)
    kw = dict(nLiar=N // 4) if fault != 'clean' else {}
    level = f'{arm}|{fault}' + (f'|q{q}' if q != 2 else '')
    return clueCell(study, level, N, seed, runner, s=CLUE_SETTINGS, channel='blackboard', world=w, **kw)


STAGE1_CELLS = [(a, f) for a in ('plain', 'confirm', 'colony', 'colonyGuards', 'record') for f in ('independent',)] + \
               [(a, 'error') for a in ('plain', 'colony')] + [(a, 'clean') for a in ('plain', 'confirm', 'colony', 'colonyGuards')]


def colonyPilotGrid():
    "N 16, plain and colony, clean and with independent liars, the pilot seed (not analysed)."
    return [colonyCell('colonyPilot', a, 16, PILOT_SEED, f) for a in ('plain', 'colony') for f in ('clean', 'independent')]


def stage1Grid(seeds=COLONY_SEEDS, sizes=STAGE1_N):
    "Study colony1: STAGE1_CELLS (11 per seed and size) at N 8 and 16."
    return [colonyCell('colony1', a, N, s, f) for a, f in STAGE1_CELLS for N in sizes for s in seeds]


def stage2Grid(arm, seeds=COLONY_SEEDS, seeds64=None, sizes=STAGE2_N):
    "Study colony2: plain and the chosen arm, clean, with independent and with colluding liars; seeds64 at N 64 (cut)."
    sd = lambda N: seeds64 if seeds64 is not None and N == 64 else seeds
    return [colonyCell('colony2', a, N, s, f) for a in ('plain', arm) for f in ('clean', 'independent', 'collude')
            for N in sizes for s in sd(N)]


def part0Grid(seeds=PART0_SEEDS, sizes=PART0_N):
    "Study colony0 (numpy): PART0_ARMS x faults (none and three kinds) x q in {2, 3} x N 4-128."
    return [colonyCell('colony0', a, N, s, f, 'numpy', q) for a in PART0_ARMS for f in FAULTS18 for q in (2, 3)
            for N in sizes for s in seeds]


def _short(cells, turns):
    return [dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=turns)) for c in cells]


def stage1Quick(turns=3):
    "Stage 1's cells at seed 1800, a few turns (the fake-client pipeline check)."
    return _short(stage1Grid([1800]), turns)

## Estimates, cost rules, go/no-go

In [ ]:
#| export
def colonyEstimate(cells):
    "fixEstimate with the pilot's cells as cached (no earlier phase's cell is reused: the deal differs at r = 2)."
    return fixEstimate(cells, colonyPilotGrid())


def costRule1(perCall, plan=None):
    "Stage 1 (FR-A10): projected = new calls x perCall: up to $35 run 8 seeds; up to $50 run 6; else stop and ask."
    r = (plan or COLONY_PLAN)['costRule1']
    proj = colonyEstimate(stage1Grid())['newCalls'] * perCall
    verdict = 'run' if proj <= r['run'] else 'cut' if proj <= r['cut'] else 'stop'
    seeds = dict(run=COLONY_SEEDS, cut=COLONY_SEEDS[:r['cutSeeds']]).get(verdict)
    return dict(verdict=verdict, projected=proj, seeds=seeds, cells=stage1Grid(seeds) if seeds else None)


def costRule2(perCall, arm, plan=None):
    "Stage 2: above $90 projected, N 64 runs 5 seeds (1800-1804)."
    r = (plan or COLONY_PLAN)['costRule2']
    full = stage2Grid(arm)
    proj = colonyEstimate(full)['newCalls'] * perCall
    if proj <= r['budget']:
        return dict(verdict='run', projected=proj, cells=full)
    cut = stage2Grid(arm, seeds64=COLONY_SEEDS[:r['cutSeeds64']])
    return dict(verdict='cut', projected=colonyEstimate(cut)['newCalls'] * perCall, cells=cut)


def goNoGo(rows, alpha=0.05, sizes=STAGE1_N):
    """FR-A11: the chosen arm (colony or colonyGuards, the higher mean honest pooled share with independent liars at the
    larger Stage 1 size; colony on a tie); go if its one-sided sign-flip p at that size is below alpha and its mean
    advantage there is not below its mean advantage at the smaller size."""
    lo, hi = sizes
    mean = lambda a: np.nanmean(list(cellVals(rows, a, 'independent', 'honestPooled', [hi]).values()) or [NAN])
    arm = 'colonyGuards' if mean('colonyGuards') > mean('colony') else 'colony'
    d = {N: pairs(rows, arm, 'honestPooled', sizes=[N]) for N in sizes}
    p = signFlipExact(list(d[hi].values()))
    a8, a16 = (float(np.mean(list(d[N].values()))) if d[N] else NAN for N in sizes)
    go = bool(p < alpha and a16 >= a8)
    return dict(verdict='go' if go else 'stop', arm=arm, p=p, adv=dict(zip(sizes, (a8, a16))))

## Plans

In [ ]:
#| export
COLONY_PLAN = dict(
    version=1, frozen=False, model=HAIKU, prices=PRICES, world=WORLD18, turns=40, channel='blackboard', memory='full',   # as Phases 11-15 (the cost estimate uses their prices)
    armsWorld={a: COLONY_ARMS[a] for a in COLONY_ARMS}, faults=dict(nLiar='N // 4', slots='0 .. N // 4 - 1',
        independent='ClueLiar', collude='ColludingLiar', error='MistakenAgent (p 0.3)'),
    seeds=dict(stage1=COLONY_SEEDS, pilot=PILOT_SEED, part0=PART0_SEEDS), sizes=STAGE1_N,
    stage1=dict(sizes=STAGE1_N, cells=[f'{a}|{f}' for a, f in STAGE1_CELLS]),
    pilot=dict(N=16, arms=['plain', 'colony'], faults=['clean', 'independent'], bands=dict(invalid=0.05),
               liarCost='stop and ask if the plain board shows no honest loss'),
    costRule1=dict(run=35.0, cut=50.0, cutSeeds=6, otherwise='stop and ask',
                   projection="Stage 1's new calls x the pilot's measured cost per call (N 16)"),
    costRule2=dict(budget=90.0, cutSeeds64=5),
    hypotheses=dict(
        H18_1='colony - plain, honestPooled, independent liars, 16 pairs (seed x N): one-sided exact sign-flip > 0',
        H18_2='plain - colony, contamination, independent liars, 16 pairs: > 0',
        H18_3='0.05 + (colony - plain) pooledShare clean, 16 pairs: > 0 (non-inferiority, margin 0.05)',
        H18_4='colonyGuards - colony, honestPooled, independent liars, 16 pairs: > 0',
        H18_5='Stage 1 (exploratory): per seed, advantage at 16 - at 8 (colony), 8 seeds; Stage 2 (confirmatory, '
              'COLONY2_PLAN): per seed, the slope over log2 N of the chosen arm\'s advantage at 8-64, one-sided sign-flip',
        H18_6='secondary: colony - plain, honestPooled, mistaken members, 16 pairs'),
    holm=['H18_1', 'H18_2', 'H18_3', 'H18_4'], alpha=0.05, margin=0.05, arms=list(COLONY_ARMS),
    secondary='colony - confirm; loss recovered and share of the record\'s; goal drift (freeRiding, misreport), the action mix and '
              'efficiency (team welfare per 1k output tokens and per dollar) per arm; mechanism: suppression, relay, '
              'reach, confirmations and contests per false value',
    goNoGo='goNoGo: the chosen arm (colony or colonyGuards, the higher mean honestPooled with liars at N 16; colony on a '
           'tie); go if its one-sided sign-flip p at N 16 < 0.05 and its mean advantage at 16 >= at 8; else stop',
    forecast='colony_predictions.json committed before the pilot; a sign counts when the Haiku interval excludes 0 on the '
             'predicted side; spanning 0 is no call',
    report=['honestPooled', 'pooledShare', 'contamination', 'reach', 'suppression', 'relay', 'freeRiding', 'misreport',
            'openShare', 'submitShare', 'shareShare', 'waitShare'],
    tracing='colony_trace: outcome-based contamination (a wrong honest answer agreeing with a false value seen before it); '
            'on Phase 14 cells never above Phase 15\'s exposure-based measure, same trend in N (AC-6, revised after measuring)',
    nboot=2000, ci=0.95, exploratory=True)
COLONY2_PLAN = dict(version=1, frozen=False, arm=None, capRaised=False, cap=None, priceRatio=2.0,   # N 32-64 cost per call ~2x N 16's (Phase 14)
                     seeds=COLONY_SEEDS, sizes=STAGE2_N,
                    faults=['clean', 'independent', 'collude'], costRule2=COLONY_PLAN['costRule2'],
                    trend='per seed, the least-squares slope over log2 N (8, 16, 32, 64) of the chosen arm - plain in '
                          'honestPooled with independent liars; one-sided sign-flip over the seeds; the arm was chosen on '
                          'Stage 1 data (reported)', collude='descriptive: colluding against independent liars per N',
                    exploratory=True)

## Synthetic rows (tests)

In [ ]:
#| export
def syntheticRows18(effects=None, seeds=range(1800, 1808), sizes=(8, 16), noise=0.02, seed=0):
    """Rows with known effects on honestPooled, contamination and pooledShare: effects {arm: (attack gain, clean change,
    contamination change, slope per log2 N)} against plain (0.4 clean, 0.25 with liars, 0.3 contamination)."""
    eff = dict(plain=(0, 0, 0, 0), confirm=(0.02, 0, -0.02, 0), colony=(0.08, -0.01, -0.1, 0.02), colonyGuards=(0.11, -0.02, -0.12, 0.02),
               record=(0.1, 0, -0.1, 0)) if effects is None else effects
    rng, rows = np.random.default_rng(seed), []
    for a, (g, c, ct, sl) in eff.items():
        for f in ('clean', 'independent', 'error'):
            if (f == 'clean' and a == 'record') or (f == 'error' and a not in ('plain', 'colony')):
                continue
            for s in seeds:
                for N in sizes:
                    e = rng.normal(0, noise)
                    hp = (0.4 + c if f == 'clean' else 0.25 + g + sl * np.log2(N / sizes[0])) + e
                    rows.append(dict(level=f'{a}|{f}', seed=s, N=N, honestPooled=hp, pooledShare=hp, welfareHonest=hp,
                                     welfare=hp, tEnd=40, outputTokens=1000 * N, cost=0.01 * N,
                                     contamination=NAN if f == 'clean' else 0.3 + ct + e, freeRiding=0.1, misreport=1.0))
    return rows

## Tests

In [ ]:
import json
from explRL.colony_grid import *
from explRL.coop_grid import coopHash
from explRL.llm_runs import LLMConfig

g = stage1Grid()
assert len(g) == 11 * 2 * 8 == 176 and len({(c['level'], c['N'], c['seed']) for c in g}) == 176
for c in g:
    cfg = LLMConfig.fromDict(c['cfg'])
    arm, fault = c['level'].split('|')
    assert cfg.world['r'] == 2 and cfg.turns == 40 and cfg.channel == 'blackboard'
    assert cfg.nLiar == (0 if fault == 'clean' else c['N'] // 4)
    assert cfg.world.get('guards', 0) == (c['N'] // 4 if arm == 'colonyGuards' else 0)
    assert cfg.world.get('fault', 'independent') == ('error' if fault == 'error' else 'independent')
assert {c['seed'] for c in g} == set(COLONY_SEEDS)
assert len(colonyPilotGrid()) == 4 and all(c['seed'] == PILOT_SEED and c['N'] == 16 for c in colonyPilotGrid())
s2 = stage2Grid('colony')
assert len(s2) == 2 * 3 * 2 * 8 and len(stage2Grid('colony', seeds64=COLONY_SEEDS[:5])) == 2 * 3 * (8 + 5)
assert {c['cfg']['world'].get('fault') for c in s2 if c['level'].endswith('collude')} == {'collude'}
p0 = part0Grid(seeds=[3000])
assert len(p0) == 7 * 4 * 2 * 6 and all(c['runner'] == 'numpy' for c in p0)
assert all(c['cfg']['world']['r'] == 3 for c in p0 if c['level'].endswith('|q3'))
# estimates and the cost rules
e = colonyEstimate(stage1Grid())
calls = lambda N: 4 * N * 40 + 7 * (N - N // 4) * 40
assert e['newCalls'] == 8 * (calls(8) + calls(16)) == 8 * (2960 + 5920)
assert costRule1(30 / e['newCalls'])['verdict'] == 'run' and costRule1(40 / e['newCalls'])['verdict'] == 'cut'
assert costRule1(40 / e['newCalls'])['seeds'] == COLONY_SEEDS[:6] and costRule1(60 / e['newCalls'])['verdict'] == 'stop'
e2 = colonyEstimate(stage2Grid('colony'))['newCalls']
assert costRule2(80 / e2, 'colony')['verdict'] == 'run' and costRule2(120 / e2, 'colony')['verdict'] == 'cut'
# go / no-go: both verdicts and the tie rule
rows = syntheticRows18()
r = goNoGo(rows)
assert r['verdict'] == 'go' and r['arm'] == 'colonyGuards' and r['p'] == 1 / 256
assert goNoGo(syntheticRows18({a: (0.0, 0, 0, 0) for a in ('plain', 'colony', 'colonyGuards')}))['verdict'] == 'stop'
tie = syntheticRows18({'plain': (0, 0, 0, 0), 'colony': (0.1, 0, 0, 0), 'colonyGuards': (0.1, 0, 0, 0)}, noise=0.0)
assert goNoGo(tie)['arm'] == 'colony'
shrink = syntheticRows18({'plain': (0, 0, 0, 0), 'colony': (0.1, 0, 0, -0.05), 'colonyGuards': (0, 0, 0, 0)}, noise=0.0)
assert goNoGo(shrink)['verdict'] == 'stop'                                   # advantage smaller at 16 than at 8
json.dumps(COLONY_PLAN); json.dumps(COLONY2_PLAN)
assert not COLONY_PLAN['frozen'] and coopHash(COLONY_PLAN)
assert all(c['cfg']['memory'] == COLONY_PLAN['memory'] for c in stage1Grid() + stage2Grid('colony') + colonyPilotGrid())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()